In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Algorithms
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb


In [ ]:
# Load dataset
df = pd.read_csv("/content/symptom_Description.csv")

In [ ]:
# Basic info
print(df.shape)

In [ ]:
print(df.columns)

In [ ]:
print(df.head())

In [ ]:
print(df.info())

In [ ]:
print(df.describe())

In [ ]:
 # Check missing values
print(df.isnull().sum())

In [ ]:
# Check duplicates
print(df.duplicated().sum())

In [ ]:
# Remove duplicates if any
data = df.drop_duplicates()

In [ ]:
#Encode Categorical Data
# Most features are likely text (symptoms), disease is target
le = LabelEncoder()
data['Disease'] = le.fit_transform(data['Disease'])  # encode target variable

# If symptoms are separate columns, encode them or use One-Hot encoding
# Example for a symptom column:
# data['Symptom_1'] = le.fit_transform(data['Symptom_1'])


In [ ]:
#Feature & Target Separation
X = data.drop('Disease', axis=1)  # features
y = data['Disease']               # target

In [ ]:
#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
#Feature Scaling (Optional for SVM & Logistic Regression)
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer to convert text descriptions into numerical features
tfidf_vectorizer = TfidfVectorizer(max_features=1000) # Limiting features for demonstration

# Fit and transform the 'Description' text data for training and test sets
X_train_transformed = tfidf_vectorizer.fit_transform(X_train['Description']).toarray()
X_test_transformed = tfidf_vectorizer.transform(X_test['Description']).toarray()

# Now apply StandardScaler to the numerical, vectorized features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_transformed)
X_test_scaled = scaler.transform(X_test_transformed)

In [ ]:
#Train Models
#SVM
svm_model = SVC()
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)


In [ ]:
#Logistic Regression
log_model = LogisticRegression(max_iter=500)
log_model.fit(X_train_scaled, y_train)
y_pred_log = log_model.predict(X_test_scaled)


In [ ]:
#Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

In [ ]:
#XGBoost
# Create a temporary LabelEncoder for the current train/test split to make labels contiguous
temp_le = LabelEncoder()
y_train_encoded = temp_le.fit_transform(y_train)

xgb_model = xgb.XGBClassifier(
    objective='multi:softmax', # Explicitly setting objective for multi-class
    num_class=len(temp_le.classes_), # Number of classes *in the current training split*
    use_label_encoder=False   # Recommended for newer versions
    # Removed eval_metric='mlogloss' to avoid strict label contiguity check
)
xgb_model.fit(X_train_scaled, y_train_encoded)
y_pred_xgb_encoded = xgb_model.predict(X_test_scaled)

# Inverse transform predictions back to original labels for evaluation
y_pred_xgb = temp_le.inverse_transform(y_pred_xgb_encoded)

In [ ]:
def evaluate_model(y_test, y_pred, model_name):
    print(f"--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Evaluate all
evaluate_model(y_test, y_pred_svm, "SVM")

In [ ]:
evaluate_model(y_test, y_pred_log, "Logistic Regression")

In [ ]:
evaluate_model(y_test, y_pred_rf, "Random Forest")

In [ ]:
evaluate_model(y_test, y_pred_xgb, "XGBoost")

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues')
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Save the Random Forest model to a file
import pickle

with open('disease_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("Random Forest model saved as 'disease_model.pkl'")